# 05c: laterality classifier (which knee is it)

Image right is always patient left, so in a left knee the fibula (lateral) is on the image
right, and in a right knee on the image left. A small ResNet-18 learns this from three
posterior coronal slices.

Training labels: DICOM tag or scanner geometry (3309 studies). 1098 studies are unknown,
mostly GE, Toshiba and Canon, which have few tags, so the check that matters is on
unseen sites.

Checks:
- 5-fold out-of-fold accuracy, folds grouped by site (language x vendor).
- On the unknown studies: agreement with the side named in the report (report side is only
  used for this check, never for training).

Augmentation: horizontal flip with the label swapped (a mirrored left knee looks like a
right knee).

Outputs: lat_model.pt (for 05b), laterality_final.csv (side for every train study).
Settings: GPU (one is enough), Internet on (pretrained weights).
Inputs: competition data, outputs of 01 (study_inventory.csv) and 02b (cache,
laterality.csv), utility script knee_model.

# Cell 1: inputs and coronal slices into RAM

In [ ]:
import os, sys, glob, json, time, re
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
OUT = Path("/kaggle/working")

def find_all(pattern):
    hits = glob.glob(f"/kaggle/usr/lib/**/{pattern}", recursive=True)
    for top in os.listdir("/kaggle/input"):
        if top != "competitions":
            hits += glob.glob(f"/kaggle/input/{top}/**/{pattern}", recursive=True)
    return sorted(hits)

def find_one(pattern):
    hits = find_all(pattern)
    print(f"{pattern}: {hits[:3]}")
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]

KM_DIR = os.path.dirname(find_one("knee_model.py"))
CACHE_DIR = [d for d in find_all("cache") if os.path.isdir(d)][0]
LAT_CSV = find_one("laterality.csv")
INV_CSV = find_one("study_inventory.csv")
sys.path.insert(0, KM_DIR)
import knee_model as km
print("lat helpers attached:", hasattr(km, "lat_input"))

lat = pd.read_csv(LAT_CSV)
inv = pd.read_csv(INV_CSV)
df = lat[["StudyInstanceUID", "side", "side_src", "vendor"]].merge(
    inv[["StudyInstanceUID", "lang"]], on="StudyInstanceUID", how="left").reset_index(drop=True)
df["site"] = df["lang"].fillna("na") + "_" + df["vendor"].fillna("na")
print(df["side_src"].value_counts().to_string())

def load_cor(sid):
    try:
        with np.load(f"{CACHE_DIR}/{sid}.npz") as z:
            return km.lat_input({"cor": z["cor"]}) if "cor" in z.files else None
    except Exception:
        return None

t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    arrs = list(ex.map(load_cor, df["StudyInstanceUID"]))
ok = np.array([a is not None for a in arrs])
X = np.zeros((len(df), 3, km.SIZE, km.SIZE), np.uint8)
for i, a in enumerate(arrs):
    if a is not None:
        X[i] = a
del arrs
print(f"coronal inputs: {ok.sum()} of {len(df)} in {time.time() - t0:.0f}s, {X.nbytes / 1e9:.2f} GB in RAM")

# Cell 2: side named in the report (check only)

In [ ]:
SIDE = {
    "L": r"\b(left|izquierd[ao]|links|linke\w*|gauche|lijev\w*)\b",
    "R": r"\b(right|derech[ao]|rechts|rechte\w*|droite?|desn\w*)\b",
}

def side_of(text):
    l = re.search(SIDE["L"], text) is not None
    r = re.search(SIDE["R"], text) is not None
    return "L" if l and not r else "R" if r and not l else None

reports = pd.read_csv(f"{ROOT}/train.csv", usecols=["StudyInstanceUID", "Report"]).set_index("StudyInstanceUID")["Report"]
df["rep_side"] = df["StudyInstanceUID"].map(reports.str.lower().map(side_of))

known = df["side"].isin(["L", "R"]).values & ok
Y = (df["side"] == "L").astype("float32").values
print("known side:", int(known.sum()), " unknown:", int((~known & ok).sum()))
k = df[known & df["rep_side"].notna().values]
print(f"report regex vs known side: {(k['side'] == k['rep_side']).mean():.4f} agreement on {len(k)} studies")

# Cell 3: 5-fold check, folds grouped by site

In [ ]:
import torch
import torch.nn.functional as F

def train_lat(tr_idx, epochs=6, bs=64, lr=3e-4, seed=0):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = km.make_lat_net(pretrained=True).cuda()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * (len(tr_idx) // bs),
                                                pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda")
    model.train()
    for ep in range(epochs):
        perm, tot, n = rng.permutation(tr_idx), 0.0, 0
        for i in range(0, len(perm) - bs + 1, bs):
            b = perm[i:i + bs]
            x = torch.from_numpy(X[b]).cuda(non_blocking=True).float() / 255.0
            oy, ox = rng.integers(0, km.SIZE - km.CROP + 1, 2)
            x = x[:, :, oy:oy + km.CROP, ox:ox + km.CROP]
            y = torch.from_numpy(Y[b]).cuda()
            flip = torch.rand(len(b), device="cuda") < 0.5
            x = torch.where(flip[:, None, None, None], x.flip(-1), x)
            y = torch.where(flip, 1 - y, y)
            with torch.autocast("cuda", dtype=torch.float16):
                logit = model(km.lat_norm(x)).float().squeeze(1)
            loss = F.binary_cross_entropy_with_logits(logit, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot, n = tot + loss.item() * len(b), n + len(b)
    print(f"  last epoch train loss {tot / n:.4f}")
    model.eval()
    return model

def predict(model, idx):
    return np.concatenate([km.lat_predict(model, X[idx[i:i + 256]], "cuda") for i in range(0, len(idx), 256)])

lab_idx = np.where(known)[0]
oof = np.full(len(df), np.nan)
t0 = time.time()
for f, (tr, va) in enumerate(GroupKFold(n_splits=5).split(lab_idx, groups=df.loc[lab_idx, "site"])):
    m = train_lat(lab_idx[tr], seed=f)
    oof[lab_idx[va]] = predict(m, lab_idx[va])
    acc = ((oof[lab_idx[va]] > 0.5) == (Y[lab_idx[va]] == 1)).mean()
    print(f"fold {f}: val {len(va)} studies, sites {sorted(df.loc[lab_idx[va], 'site'].unique())[:6]}, acc {acc:.4f}")
    del m
    torch.cuda.empty_cache()

k = df.loc[lab_idx].assign(p=oof[lab_idx])
k["correct"] = (k["p"] > 0.5) == (k["side"] == "L")
print(f"\nout-of-fold accuracy {k['correct'].mean():.4f} on {len(k)} studies ({(time.time() - t0) / 60:.1f} min)")
print(k.groupby("vendor")["correct"].agg(["mean", "size"]).round(4).to_string())
print(k.groupby("side_src")["correct"].agg(["mean", "size"]).round(4).to_string())
print("uncertain (0.1 < p < 0.9):", round(float(((k["p"] > 0.1) & (k["p"] < 0.9)).mean()), 4))

# Cell 4: final model, unknown studies, save

In [ ]:
final = train_lat(lab_idx, seed=42)
unk_idx = np.where(~known & ok)[0]
p_unk = predict(final, unk_idx)

u = df.loc[unk_idx].assign(p_left=p_unk, pred=np.where(p_unk > 0.5, "L", "R"))
chk = u[u["rep_side"].notna()]
print(f"unknown studies: {len(u)}; with a side in the report: {len(chk)}; "
      f"classifier agrees with report: {(chk['pred'] == chk['rep_side']).mean():.4f}")
print(chk.assign(agree=chk["pred"] == chk["rep_side"]).groupby("vendor")["agree"].agg(["mean", "size"]).round(4).to_string())
print("unknown studies by vendor:", u["vendor"].value_counts().to_dict())
print("uncertain (0.1 < p < 0.9):", round(float(((p_unk > 0.1) & (p_unk < 0.9)).mean()), 4))

df["p_left"] = np.nan
df.loc[unk_idx, "p_left"] = p_unk
df["side_final"] = df["side"]
df.loc[unk_idx, "side_final"] = u["pred"].values
df["side_src_final"] = df["side_src"]
df.loc[unk_idx, "side_src_final"] = "classifier"
print(df["side_src_final"].value_counts().to_string())
print(df["side_final"].value_counts(dropna=False).to_string())

torch.save({"state_dict": final.state_dict(), "arch": "resnet18", "fracs": list(km.LAT_FRACS)}, OUT / "lat_model.pt")
df[["StudyInstanceUID", "side_final", "side_src_final", "p_left"]].to_csv(OUT / "laterality_final.csv", index=False)
json.dump({"oof_acc": float(k["correct"].mean()), "oof_n": int(len(k)),
           "unknown_n": int(len(u)), "unknown_report_n": int(len(chk)),
           "unknown_report_agree": float((chk["pred"] == chk["rep_side"]).mean()) if len(chk) else None},
          open(OUT / "lat_summary.json", "w"), indent=1)
print("saved lat_model.pt, laterality_final.csv, lat_summary.json")